# Experiment Runner
Speaker / Listener × Logit / Choice × 모델 선택

**사용법**: Config 만 수정하여 사용

In [1]:
import csv
from pathlib import Path
from random import sample
from variables import *
from scoring import score_candidates

In [15]:
# ── CONFIG ──────────────────────── (여기만 수정)
MODEL       = "qwen3-8b"   # "llama3" | "qwen3"
ROLE        = "listener"  # "speaker" | "listener"
MODE        = "choice"    # "logit"   | "choice"
REPETITIONS = 10          # logit → 1 권장 (결정론적),  choice → 5~10
SHOT = "zero"  # "zero" | "one" | "few"
# ────────────────────────────────────────────────

print(f"model={MODEL}  role={ROLE}  mode={MODE}  reps={REPETITIONS}")

model=qwen3-8b  role=listener  mode=choice  reps=10


In [16]:
# ── 모델 로드 ─────────────────────────────────────
from llama_cpp import Llama

llm = Llama.from_pretrained(
    **MODEL_CONFIGS[MODEL],
    n_ctx=2048,
    n_threads=4,
    logits_all=True,
)
print(f"로드 완료: {MODEL}")

llama_model_load_from_file_impl: using device MTL0 (Apple M2) (unknown id) - 10922 MiB free
llama_model_loader: loaded meta data with 28 key-value pairs and 399 tensors from /Users/eyun/.cache/huggingface/hub/models--Qwen--Qwen3-8B-GGUF/snapshots/7c41481f57cb95916b40956ab2f0b139b296d974/./Qwen3-8B-Q4_K_M.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = qwen3
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Qwen3 8B Awq Compatible Instruct
llama_model_loader: - kv   3:                           general.finetune str              = awq-compatible-Instruct
llama_model_loader: - kv   4:                           general.basename str              = Qwen3
llama_model_loader: - kv   5:

로드 완료: qwen3-8b


In [17]:
# ── 실험 루프 ─────────────────────────────────────
base_prompt = Path(f"prompts/{ROLE}_{MODE}_{SHOT}.txt").read_text(encoding="utf-8").strip()

candidates = ADJ_CANDIDATES if ROLE == "speaker" else STATE_CANDIDATES
outer_loop = STATE_VAR      if ROLE == "speaker" else ADJECTIVES

rows  = []
total = len(SITUATIONS) * len(RELATIONSHIP_VAR) * len(outer_loop) * REPETITIONS
print(f"총 {total} rows 생성 예정")
count = 0

for situation, scenario_template in SITUATIONS.items():
    for rel in RELATIONSHIP_VAR:
        for outer in outer_loop:
            for _ in range(REPETITIONS):
                personA, personB = sample(NAME_VARIATIONS, 2)
                scenario = scenario_template.format(personA=personA, personB=personB)
                thing    = THING_KEYWORDS[situation]

                fmt = dict(
                    personA=personA, personB=personB,
                    relationship=rel, scenario=scenario, thing=thing,
                )
                if ROLE == "speaker":
                    fmt["state"] = outer
                else:
                    fmt["adjective"] = f" {outer}"   # leading space for tokenization

                prompt = base_prompt.format(**fmt)

                row = dict(
                    situation=situation, relationship=rel,
                    personA=personA, personB=personB,
                )
                row["state" if ROLE == "speaker" else "adjective"] = outer

                if MODE == "logit":
                    logits, probs, logprobs = score_candidates(llm, prompt, candidates)
                    for c in candidates:
                        k = c.strip()
                        row[f"logit_{k}"]   = logits[c]
                        row[f"prob_{k}"]    = probs[c]
                        row[f"logprob_{k}"] = logprobs[c]
                    row["pred_top1"] = max(candidates, key=lambda c: probs[c]).strip()
                else:
                    resp = llm(prompt, max_tokens=30, temperature=0.0, top_k=1, top_p=0.9)
                    row["response_text"] = resp["choices"][0]["text"].strip()

                rows.append(row)
                count += 1
                if count % 100 == 0:
                    label = f"state={outer}" if ROLE == "speaker" else f"adj={outer}"
                    print(f"  [{count}/{total}] {situation} | {rel} | {label}")

Path("results").mkdir(exist_ok=True)
out = Path(f"results/{ROLE}_{MODE}_{MODEL}_{SHOT}.csv")
with open(out, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    w.writeheader()
    w.writerows(rows)

print(f"\n완료 → {out}  ({len(rows)} rows)")

총 1000 rows 생성 예정


llama_perf_context_print:        load time =       0.37 ms
llama_perf_context_print: prompt eval time =       0.06 ms /   161 tokens (    0.00 ms per token, 2683333.33 tokens per second)
llama_perf_context_print:        eval time =    2379.10 ms /    29 runs   (   82.04 ms per token,    12.19 tokens per second)
llama_perf_context_print:       total time =    7801.16 ms /   190 tokens
llama_perf_context_print:    graphs reused =         28
Llama.generate: 42 prefix-match hit, remaining 118 prompt tokens to eval
llama_perf_context_print:        load time =       0.37 ms
llama_perf_context_print: prompt eval time =    3173.12 ms /   118 tokens (   26.89 ms per token,    37.19 tokens per second)
llama_perf_context_print:        eval time =    2406.25 ms /    29 runs   (   82.97 ms per token,    12.05 tokens per second)
llama_perf_context_print:       total time =    5597.86 ms /   147 tokens
llama_perf_context_print:    graphs reused =         28
Llama.generate: 42 prefix-match hit, remain

  [100/1000] Kuchen | Entfernte Kollegin | adj=schrecklich


llama_perf_context_print:        load time =       0.37 ms
llama_perf_context_print: prompt eval time =    4021.24 ms /   116 tokens (   34.67 ms per token,    28.85 tokens per second)
llama_perf_context_print:        eval time =    2440.62 ms /    29 runs   (   84.16 ms per token,    11.88 tokens per second)
llama_perf_context_print:       total time =    6473.46 ms /   145 tokens
llama_perf_context_print:    graphs reused =         28
Llama.generate: 42 prefix-match hit, remaining 119 prompt tokens to eval
llama_perf_context_print:        load time =       0.37 ms
llama_perf_context_print: prompt eval time =    4034.27 ms /   119 tokens (   33.90 ms per token,    29.50 tokens per second)
llama_perf_context_print:        eval time =    2396.20 ms /    29 runs   (   82.63 ms per token,    12.10 tokens per second)
llama_perf_context_print:       total time =    6445.84 ms /   148 tokens
llama_perf_context_print:    graphs reused =         28
Llama.generate: 42 prefix-match hit, remainin

  [200/1000] Kuchen | Gefürchtete Chefin | adj=schrecklich


...

  [800/1000] Theater | Gefürchtete Chefin | adj=schrecklich


llama_perf_context_print:        load time =       0.37 ms
llama_perf_context_print: prompt eval time =    6365.11 ms /   120 tokens (   53.04 ms per token,    18.85 tokens per second)
llama_perf_context_print:        eval time =    3856.27 ms /    29 runs   (  132.97 ms per token,     7.52 tokens per second)
llama_perf_context_print:       total time =   10238.64 ms /   149 tokens
llama_perf_context_print:    graphs reused =         28
Llama.generate: 42 prefix-match hit, remaining 118 prompt tokens to eval
llama_perf_context_print:        load time =       0.37 ms
llama_perf_context_print: prompt eval time =    9890.81 ms /   118 tokens (   83.82 ms per token,    11.93 tokens per second)
llama_perf_context_print:        eval time =    3181.39 ms /    29 runs   (  109.70 ms per token,     9.12 tokens per second)
llama_perf_context_print:       total time =   13117.39 ms /   147 tokens
llama_perf_context_print:    graphs reused =         28
Llama.generate: 45 prefix-match hit, remainin

  [900/1000] Gitarre | Entfernte Kollegin | adj=schrecklich


llama_perf_context_print:        load time =       0.37 ms
llama_perf_context_print: prompt eval time =    6543.30 ms /   113 tokens (   57.91 ms per token,    17.27 tokens per second)
llama_perf_context_print:        eval time =    2912.22 ms /    29 runs   (  100.42 ms per token,     9.96 tokens per second)
llama_perf_context_print:       total time =    9470.95 ms /   142 tokens
llama_perf_context_print:    graphs reused =         28
Llama.generate: 42 prefix-match hit, remaining 127 prompt tokens to eval
llama_perf_context_print:        load time =       0.37 ms
llama_perf_context_print: prompt eval time =    6864.53 ms /   127 tokens (   54.05 ms per token,    18.50 tokens per second)
llama_perf_context_print:        eval time =    2975.72 ms /    29 runs   (  102.61 ms per token,     9.75 tokens per second)
llama_perf_context_print:       total time =    9865.00 ms /   156 tokens
llama_perf_context_print:    graphs reused =         28
Llama.generate: 42 prefix-match hit, remainin

  [1000/1000] Gitarre | Gefürchtete Chefin | adj=schrecklich

완료 → results/listener_choice_qwen3-8b_zero.csv  (1000 rows)
